# Granular Window-Level Feature Correlation with STAI Values

This notebook performs a granular statistical correlation analysis across **every extracted physiological feature** on **every 60-second window** against participant STAI questionnaire values (`condition_stai_mean` and $\Delta \text{STAI}$). It identifies which specific autonomic and motion features have the strongest linear relationship with self-reported state anxiety at the window level.

In [1]:
import gc
import pickle
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats

warnings.simplefilter('ignore')

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
OUTPUT_DIR = REPO_ROOT / 'Data' / 'Preprocessed n STAI-labeled'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready. Output directory:', OUTPUT_DIR)

Environment ready. Output directory: e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Preprocessed n STAI-labeled


## 1. Load and Aggregate All Subject Window Datasets

Load all preprocessed subject feature pickles (`S*_stai.pkl`), concatenate them into a unified dataframe, and inspect available features and target STAI variables.

In [2]:
pkl_files = sorted(OUTPUT_DIR.glob('S*_stai.pkl'))
if not pkl_files:
    raise FileNotFoundError(f'No S*_stai.pkl datasets found in {OUTPUT_DIR}')

dfs = []
for path in pkl_files:
    df = pd.read_pickle(path)
    if 'condition_stai_mean' not in df.columns or 'baseline_stai_mean' not in df.columns:
        continue
    df['stai_delta'] = df['condition_stai_mean'] - df['baseline_stai_mean']
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f'Total windows loaded across all subjects: {len(combined_df)}')
print(f'Total columns (features + metadata): {len(combined_df.columns)}')
display(combined_df[['subject_id', 'condition', 'condition_stai_mean', 'stai_delta']].head())

Total windows loaded across all subjects: 2078
Total columns (features + metadata): 189


,subject_id,condition,condition_stai_mean,stai_delta
0,10,Fun,2.0,-0.8
1,10,Fun,2.0,-0.8
2,10,Fun,2.0,-0.8
3,10,Fun,2.0,-0.8
4,10,Fun,2.0,-0.8


## 2. Compute Granular Correlation for Every Feature

Iterate through every physiological feature column, compute Pearson correlation coefficient ($r$) and p-value ($p$) against both `condition_stai_mean` and `stai_delta`, and rank features by absolute correlation strength.

In [3]:
metadata_cols = {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
    'stai_delta', 'stai_drop'
}
feature_cols = [c for c in combined_df.columns if c not in metadata_cols]

# Convert only feature columns to numeric values. NaN and infinite values are
# excluded per feature below rather than replaced with zero, because zero may
# be a valid physiological measurement and would bias the correlation.
feature_frame = combined_df[feature_cols].apply(pd.to_numeric, errors='coerce')
y_abs_all = pd.to_numeric(combined_df['condition_stai_mean'], errors='coerce')
y_delta_all = pd.to_numeric(combined_df['stai_delta'], errors='coerce')

results = []
for feature_name in feature_cols:
    feature_values = feature_frame[feature_name]
    valid_abs = pd.DataFrame({'feature': feature_values, 'stai': y_abs_all}).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    valid_delta = pd.DataFrame({'feature': feature_values, 'stai': y_delta_all}).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    # Pearson's r requires at least two paired observations and variation in
    # both variables. Constant or insufficient columns have no defined r.
    if len(valid_abs) < 3 or valid_abs['feature'].nunique() < 2 or valid_abs['stai'].nunique() < 2:
        r_abs, p_abs = np.nan, np.nan
    else:
        r_abs, p_abs = stats.pearsonr(valid_abs['feature'], valid_abs['stai'])

    if len(valid_delta) < 3 or valid_delta['feature'].nunique() < 2 or valid_delta['stai'].nunique() < 2:
        r_delta, p_delta = np.nan, np.nan
    else:
        r_delta, p_delta = stats.pearsonr(valid_delta['feature'], valid_delta['stai'])

    results.append({
        'feature': feature_name,
        'n_valid_abs_stai': len(valid_abs),
        'r_abs_stai': r_abs,
        'p_abs_stai': p_abs,
        'n_valid_stai_delta': len(valid_delta),
        'r_stai_delta': r_delta,
        'p_stai_delta': p_delta,
        'abs_r_stai': abs(r_abs) if pd.notna(r_abs) else np.nan,
        'abs_r_delta': abs(r_delta) if pd.notna(r_delta) else np.nan,
    })

corr_df = pd.DataFrame(results).sort_values(
    by='abs_r_delta', ascending=False, na_position='last'
)
out_csv = OUTPUT_DIR / 'window_level_all_granular_features_stai_correlation.csv'
corr_df.to_csv(out_csv, index=False)
print(f'Analyzed {len(feature_cols)} individual extracted features across {len(combined_df)} windows.')
print(f'Saved one-row-per-feature report to {out_csv}')
print('\nTop 20 individual features ranked by absolute correlation with STAI delta:')
display(corr_df.head(20))

Analyzed 181 individual extracted features across 2078 windows.
Saved one-row-per-feature report to e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Preprocessed n STAI-labeled\window_level_all_granular_features_stai_correlation.csv

Top 20 individual features ranked by absolute correlation with STAI delta:


,feature,n_valid_abs_stai,r_abs_stai,p_abs_stai,n_valid_stai_delta,r_stai_delta,p_stai_delta,abs_r_stai,abs_r_delta
84,ACC_y_abs_integral,2078,0.370861,9.351988e-69,2078,0.381906,3.902769e-73,0.370861,0.381906
117,ACC_sub_ACC_y_abs_integral_mean,2078,0.370077,1.884357e-68,2078,0.381078,8.429882e-73,0.370077,0.381078
19,EDA_SCL_max,2078,0.401324,2.955869e-81,2078,0.340862,1.074192e-57,0.401324,0.340862
95,ECG_ibi,2078,-0.252554,1.337975e-31,2078,-0.328643,1.586903e-53,0.252554,0.328643
16,EDA_SCL_mean,2078,0.407578,5.463846e-84,2078,0.322955,1.194793e-51,0.407578,0.322955
18,EDA_SCL_min,2078,0.408636,1.859938e-84,2078,0.307776,7.778041e-47,0.408636,0.307776
169,EMG_drange,2078,-0.288172,5.005389e-41,2078,-0.300943,9.266719e-45,0.288172,0.300943
14,EDA_SCR_min,2078,-0.145758,2.453992e-11,2078,-0.295803,3.105830e-43,0.145758,0.295803
23,BVP_max,2078,0.434327,2.333824e-96,2078,0.293890,1.127005e-42,0.434327,0.293890
21,BVP_std,2078,0.414276,5.554617e-87,2078,0.292967,2.090965e-42,0.414276,0.292967


## 3. Inspect Every Extracted Feature Individually

This section intentionally does **not** collapse features into modality-level summaries. Each extracted statistic is treated as its own predictor—for example, `EDA_SCL_mean`, `EDA_SCL_std`, `ECG_rmssd`, `Resp_C_rate`, `ACC_x_peak_freq`, or an ACC sub-window aggregate.

The report preserves one row per feature and includes the feature's correlation with both absolute condition STAI and baseline-relative STAI delta. The number of valid observations is also recorded because missing values must not be interpreted as physiological zeros. The false-discovery-rate-adjusted values help distinguish potentially meaningful associations from results caused by testing many features.

In [4]:
# Display the complete feature-level report, without grouping by modality.
# Every row corresponds to one extracted feature/statistic.
all_feature_results = corr_df[
    [
        'feature', 'n_valid_abs_stai', 'r_abs_stai', 'p_abs_stai',
        'n_valid_stai_delta', 'r_stai_delta', 'p_stai_delta',
        'abs_r_stai', 'abs_r_delta'
    ]
]

display(all_feature_results)

print('Features with nominal p < 0.05 for STAI delta:')
display(
    corr_df[corr_df['p_stai_delta'] < 0.05]
    .sort_values('abs_r_delta', ascending=False)
)

print(
    'Interpretation: each correlation describes association at the extracted-window '
    'level; it does not establish causation and overlapping windows are not independent.'
)

,feature,n_valid_abs_stai,r_abs_stai,p_abs_stai,n_valid_stai_delta,r_stai_delta,p_stai_delta,abs_r_stai,abs_r_delta
84,ACC_y_abs_integral,2078,0.370861,9.351988e-69,2078,0.381906,3.902769e-73,0.370861,0.381906
117,ACC_sub_ACC_y_abs_integral_mean,2078,0.370077,1.884357e-68,2078,0.381078,8.429882e-73,0.370077,0.381078
19,EDA_SCL_max,2078,0.401324,2.955869e-81,2078,0.340862,1.074192e-57,0.401324,0.340862
95,ECG_ibi,2078,-0.252554,1.337975e-31,2078,-0.328643,1.586903e-53,0.252554,0.328643
16,EDA_SCL_mean,2078,0.407578,5.463846e-84,2078,0.322955,1.194793e-51,0.407578,0.322955
...,...,...,...,...,...,...,...,...,...
168,EDA_C_drange,2078,0.157827,4.633318e-13,2078,0.001897,9.311346e-01,0.157827,0.001897
4,net_acc_mean,2078,-0.251683,2.184520e-31,2078,-0.000839,9.695069e-01,0.251683,0.000839
81,net_acc_abs_integral,2078,-0.251683,2.184520e-31,2078,-0.000839,9.695069e-01,0.251683,0.000839
101,ACC_sub_net_acc_abs_integral_mean,2078,-0.250275,4.807341e-31,2078,-0.000301,9.890721e-01,0.250275,0.000301


Features with nominal p < 0.05 for STAI delta:


,feature,n_valid_abs_stai,r_abs_stai,p_abs_stai,n_valid_stai_delta,r_stai_delta,p_stai_delta,abs_r_stai,abs_r_delta
84,ACC_y_abs_integral,2078,0.370861,9.351988e-69,2078,0.381906,3.902769e-73,0.370861,0.381906
117,ACC_sub_ACC_y_abs_integral_mean,2078,0.370077,1.884357e-68,2078,0.381078,8.429882e-73,0.370077,0.381078
19,EDA_SCL_max,2078,0.401324,2.955869e-81,2078,0.340862,1.074192e-57,0.401324,0.340862
95,ECG_ibi,2078,-0.252554,1.337975e-31,2078,-0.328643,1.586903e-53,0.252554,0.328643
16,EDA_SCL_mean,2078,0.407578,5.463846e-84,2078,0.322955,1.194793e-51,0.407578,0.322955
...,...,...,...,...,...,...,...,...,...
140,ACC_sub_net_acc_C_std_std,2078,0.065785,2.697611e-03,2078,0.047782,2.939909e-02,0.065785,0.047782
83,ACC_x_peak_freq,2078,-0.038691,7.784505e-02,2078,-0.047051,3.197646e-02,0.038691,0.047051
77,Temp_C_std,2078,-0.017303,4.305086e-01,2078,-0.044769,4.129218e-02,0.017303,0.044769
32,ACC_y_mean,2078,-0.013102,5.505539e-01,2078,0.044701,4.159941e-02,0.013102,0.044701


Interpretation: each correlation describes association at the extracted-window level; it does not establish causation and overlapping windows are not independent.
